# 02. Baseline

Простые модели «из коробки» — отправная точка для всех последующих экспериментов.


In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

from src.config import REPORT_IMAGES_DIR
from src.data_loader import load_raw
from src.preprocessing import time_or_random_split
from src.modeling import fit_and_evaluate_baseline, collect_metrics
from src.utils import set_seed

set_seed()
REPORT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)


## Сплит

Используем time-based split (train 70% / val 15% / test 15%) — `start_date` присутствует, а значит есть риск временного data leakage при случайном сплите.

In [2]:
df = load_raw()
train_df, val_df, test_df = time_or_random_split(df)
print(f'train={len(train_df)}, val={len(val_df)}, test={len(test_df)}')


train=7000, val=1500, test=1500


## Модели (рубрика: без feature engineering)

- `DummyRegressor(strategy='median')` — нижняя граница.
- `LinearRegression` — линейная модель «из коробки».

Используется `fit_and_evaluate_baseline`: только удаление утечек, приведение булевых, OHE + масштабирование. **Нет** извлечения календарных признаков из `start_date` (см. `prepare_features_baseline` в `src/preprocessing.py`). Колонка `start_date` отбрасывается; сезонность частично уже есть в `quarter`, `day_of_week` и др. из датасета.


In [3]:
rows = []
_, m = fit_and_evaluate_baseline(DummyRegressor(strategy='median'), train_df, val_df, test_df, 'Dummy(median)')
rows += m
_, m = fit_and_evaluate_baseline(LinearRegression(), train_df, val_df, test_df, 'LinearRegression')
rows += m

baseline = collect_metrics(rows)
baseline


,model,split,RMSE,MAE,R2
0,LinearRegression,test,72690.920595,21931.424357,0.656384
1,Dummy(median),test,127486.300966,32116.114727,-0.056916
2,LinearRegression,val,31563.878701,14749.960284,0.809795
3,Dummy(median),val,73946.195645,18846.146740,-0.043936


In [4]:
baseline.to_csv(REPORT_IMAGES_DIR / 'baseline_metrics.csv', index=False)
print('Сохранено в report/images/baseline_metrics.csv')


Сохранено в report/images/baseline_metrics.csv


## Вывод baseline

Линейная регрессия должна заметно обыгрывать Dummy по RMSE/MAE — это подтверждает, что в данных действительно есть сигнал, связанный с фичами кампании, аудитории и поведения пользователей.
